# **ENVIROMENT INITIALIZATION**

In [3]:
# IMPORTS
# Math libraries
import numpy as np
import pandas as pd
import polars as pl

# Plots
import matplotlib.pyplot as plt

# Technical libraries
from pathlib import Path
from datetime import date, timedelta

In [4]:
# CONFIGURATION
from config import (
    PROJECT_ROOT,
    AS_OF, TARGET_START, TARGET_END,
    DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_FEATURES_DIR,
    TRAIN_PATH, CV_TARGET_PATH, CV_FEATURES_PATH,
    REPORTS_DIR, REPORTS_GMV_DIR, REPORTS_CONV_FUNL_DIR, REPORTS_FEATURES_DIR,
    WINDOWS,
    GMV_COLS, ACTIVITY_COLS, CONVERSION_COLS,
    BIN_ORDER,
    create_directories,
    validate_data_exists,
)

# Creating directories
create_directories()

# Checking for data availability
validate_data_exists()

# Configuration info
print("DATA TRANSFORMING CONFIGURATION")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"TRAIN_PATH: {TRAIN_PATH}")
print(f"CV_TARGET_PATH: {CV_TARGET_PATH}")
print(f"CV_FEATURES_PATH: {CV_FEATURES_PATH}")
print(f"AS_OF: {AS_OF}")
print(f"TARGET_START: {TARGET_START}")
print(f"TARGET_END: {TARGET_END}")
print(f"WINDOWS: {WINDOWS}")

All directories created successfully.
TRAIN_PATH: 171.80 MB
CV_TARGET_PATH: 1.32 MB

All data files are present.
DATA TRANSFORMING CONFIGURATION
PROJECT_ROOT: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech
TRAIN_PATH: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\raw\train.parquet
CV_TARGET_PATH: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\cv_target_2026-01-14.parquet
CV_FEATURES_PATH: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\features\cv_features_2026-01-14.parquet
AS_OF: 2026-01-14
TARGET_START: 2026-01-15
TARGET_END: 2026-02-13
WINDOWS: [7, 14, 30, 60, 90, 180, 365]


In [5]:
# DATA LOADING
lf = pl.scan_parquet(TRAIN_PATH)
cv_target = pl.read_parquet(CV_TARGET_PATH)

print(f"Train data loaded: {lf.collect_schema()}")
print(f"CV target loaded: {cv_target.shape}")

Train data loaded: Schema({'event_date': Date, 'user_id': Int64, 'search': Int64, 'cat': Int64, 'has_search_to_cart': Int64, 'has_search_to_ord': Int64, 'has_cat_to_cart': Int64, 'has_cat_to_ord': Int64, 'search_to_cart': Int64, 'search_to_ord': Int64, 'cat_to_cart': Int64, 'cat_to_ord': Int64, 'gmv_search': Float64, 'gmv_cat': Float64, 'to_cart': Int64, 'to_ord': Int64, 'gmv': Float64, 'searches': Int64})
CV target loaded: (250000, 2)


# **FEATURE ENGINEERING**

| Категория | Фичи | Обоснование |
|---|---|---|
| **Recency** | days_since_last_activity, days_since_last_purchase, days_since_last_order, days_since_last_cart, days_since_last_search | Пользователи с недавней активностью конвертируют лучше |
| **Frequency** | active_days_last_7/14/30/60/90d, purchase_days_last_7/14/30/60/90d, orders_last_7/14/30/60/90d, searches_last_7/14/30/60/90d | Частота — главный драйвер различий (328 vs 7 заказов) |
| **Monetary** | gmv_last_7/14/30/60/90/180/365d, avg_order_value_last_30d, avg_order_value_last_90d | GMV за разные окна, средний чек |
| **Conversion** | cart_to_order_rate, search_to_cart_rate, search_to_order_rate, cat_to_cart_rate, cat_to_order_rate | Конверсии на разных этапах воронки |
| **Channel** | search_share, cat_share, uses_both_channels_hist, gmv_search_share | Доля каналов в активности |
| **Trend** | gmv_7d / gmv_30d, gmv_30d / gmv_90d, active_days_7d / active_days_30d | Ускорение/замедление активности |
| **Seasonal** | weekday, is_weekend, week_of_month, days_to_next_salary, days_since_last_salary, days_to_holiday, is_holiday_week | Зарплатные циклы, праздники, недельная сезонность |
| **Funnel** | avg_items_per_order, avg_cart_adds_per_day, avg_searches_per_day | Интенсивность на каждом этапе |
| **Historical** | gmv_same_period_last_year (если данные позволяют) | YoY baseline |